In [0]:
spark.sql("SHOW TABLES IN churn_lakehouse.silver").display()

database,tableName,isTemporary
silver,int_customer_360,false
silver,int_customer_financial,false
silver,int_customer_transactions,false
silver,stg_crm,false
silver,stg_engagement,false
silver,stg_financial,false
silver,stg_support,false
silver,stg_transactions,false


In [0]:
# Garante que o schema gold existe

spark.sql("CREATE SCHEMA IF NOT EXISTS churn_lakehouse.gold")

#Le a tabela central da silver

df = spark.table("churn_lakehouse.silver.int_customer_360")

print(f"Linhas: {df.count():,}")
print(f"Colunas: {len(df.columns)}")
df.printSchema()

Linhas: 2,000
Colunas: 41
root
 |-- customer_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- plan: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- employees_range: string (nullable = true)
 |-- acquisition_date: date (nullable = true)
 |-- cancellation_date: date (nullable = true)
 |-- churned: boolean (nullable = true)
 |-- tenure_days: long (nullable = true)
 |-- tenure_months: double (nullable = true)
 |-- current_mrr_brl: double (nullable = true)
 |-- avg_mrr_brl: double (nullable = true)
 |-- total_revenue_brl: double (nullable = true)
 |-- total_expansion_brl: double (nullable = true)
 |-- total_contraction_brl: double (nullable = true)
 |-- mrr_drop_rate: double (nullable = true)
 |-- total_invoices: long (nullable = true)
 |-- total_billed_brl: double (nullable = true)
 |-- total_unpaid_brl: double (nullable = true)
 |-- late_payment_rate: double (nullabl

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_gold = df.withColumn(
    # Score de risco composto — combina os sinais mais fortes de churn
    # Cada componente contribui com um peso baseado em importância real
    "churn_risk_score",
    (
        # Engajamento baixo = maior risco (peso 30%)
        (1 - F.col("avg_logins_per_week") / 4.0).cast("double") * 0.30 +
        # Pagamentos atrasados (peso 25%)
        F.col("late_payment_rate") * 0.25 +
        # NPS baixo (peso 20%)
        ((10 - F.col("nps_score")) / 10.0) * 0.20 +
        # Último login recente = menor risco (peso 15%)
        (F.col("last_login_days_ago") / 30.0).cast("double") * 0.15 +
        # MRR caindo = maior risco (peso 10%)
        F.coalesce(F.col("mrr_drop_rate"), F.lit(0.0)) * 0.10
    )
).withColumn(
    # Limita entre 0 e 1
    "churn_risk_score",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("churn_risk_score")))
).withColumn(
    # Segmento de risco
    "risk_segment",
    F.when(F.col("churn_risk_score") >= 0.65, "high")
     .when(F.col("churn_risk_score") >= 0.40, "medium")
     .otherwise("low")
).withColumn(
    # MRR em risco — quanto de receita pode ser perdida
    "mrr_at_risk",
    F.when(F.col("risk_segment") == "high",   F.col("current_mrr_brl"))
     .when(F.col("risk_segment") == "medium", F.col("current_mrr_brl") * 0.4)
     .otherwise(0.0)
).withColumn(
    # Acionável recomendado baseado no perfil do cliente
    "recommended_action",
    F.when(
        (F.col("risk_segment") == "high") & (F.col("last_ticket_type") == "cancelamento"),
        "Contato urgente — intenção de cancelamento detectada"
    ).when(
        (F.col("risk_segment") == "high") & (F.col("late_payment_rate") > 0.5),
        "Oferecer renegociação ou desconto de retenção"
    ).when(
        (F.col("risk_segment") == "high") & (F.col("avg_logins_per_week") < 0.5),
        "Reengajamento — cliente inativo, agendar onboarding"
    ).when(
        F.col("risk_segment") == "high",
        "Acionar Customer Success para avaliação"
    ).when(
        F.col("risk_segment") == "medium",
        "Monitorar e enviar campanha de engajamento"
    ).otherwise(
        "Manter acompanhamento padrão"
    )
).withColumn(
    # Cohort de aquisição por trimestre
    "acquisition_cohort",
    F.concat(
        F.year("acquisition_date").cast("string"),
        F.lit("-Q"),
        F.ceil(F.month("acquisition_date") / 3).cast("string")
    )
).drop("__databricks_id")

print(f"Gold Customer 360 — {df_gold.count():,} clientes")
df_gold.groupBy("risk_segment").count().orderBy("risk_segment").display()

Gold Customer 360 — 2,000 clientes


risk_segment,count
high,8
low,1694
medium,298


In [0]:
df_gold.select(
    F.round(F.mean("churn_risk_score"), 4).alias("media"),
    F.round(F.stddev("churn_risk_score"), 4).alias("desvio_padrao"),
    F.round(F.percentile_approx("churn_risk_score", 0.25), 4).alias("p25"),
    F.round(F.percentile_approx("churn_risk_score", 0.50), 4).alias("p50"),
    F.round(F.percentile_approx("churn_risk_score", 0.75), 4).alias("p75"),
    F.round(F.percentile_approx("churn_risk_score", 0.90), 4).alias("p90"),
    F.round(F.percentile_approx("churn_risk_score", 0.95), 4).alias("p95"),
    F.round(F.min("churn_risk_score"), 4).alias("min"),
    F.round(F.max("churn_risk_score"), 4).alias("max"),
).display()

media,desvio_padrao,p25,p50,p75,p90,p95,min,max
0.2887,0.1083,0.2094,0.27,0.3537,0.4332,0.4927,0.0566,0.73


In [0]:
# Calcula os thresholds baseados na distribuição real
p70 = df_gold.approxQuantile("churn_risk_score", [0.70], 0.01)[0]
p85 = df_gold.approxQuantile("churn_risk_score", [0.85], 0.01)[0]

print(f"Threshold medium (p70): {p70:.4f}")
print(f"Threshold high   (p85): {p85:.4f}")

# Recria a segmentação com thresholds corretos
df_gold = df_gold.withColumn(
    "risk_segment",
    F.when(F.col("churn_risk_score") >= p85, "high")
     .when(F.col("churn_risk_score") >= p70, "medium")
     .otherwise("low")
).withColumn(
    "mrr_at_risk",
    F.when(F.col("risk_segment") == "high",   F.col("current_mrr_brl"))
     .when(F.col("risk_segment") == "medium", F.col("current_mrr_brl") * 0.4)
     .otherwise(0.0)
).withColumn(
    "recommended_action",
    F.when(
        (F.col("risk_segment") == "high") & (F.col("last_ticket_type") == "cancelamento"),
        "Contato urgente — intenção de cancelamento detectada"
    ).when(
        (F.col("risk_segment") == "high") & (F.col("late_payment_rate") > 0.5),
        "Oferecer renegociação ou desconto de retenção"
    ).when(
        (F.col("risk_segment") == "high") & (F.col("avg_logins_per_week") < 0.5),
        "Reengajamento — cliente inativo, agendar onboarding"
    ).when(
        F.col("risk_segment") == "high",
        "Acionar Customer Success para avaliação"
    ).when(
        F.col("risk_segment") == "medium",
        "Monitorar e enviar campanha de engajamento"
    ).otherwise(
        "Manter acompanhamento padrão"
    )
)

# Verifica distribuição final
df_gold.groupBy("risk_segment").agg(
    F.count("*").alias("clientes"),
    F.round(F.sum("mrr_at_risk"), 2).alias("mrr_at_risk_brl"),
    F.round(F.mean("churn_risk_score"), 4).alias("score_medio")
).orderBy("risk_segment").display()

Threshold medium (p70): 0.3312
Threshold high   (p85): 0.3962


risk_segment,clientes,mrr_at_risk_brl,score_medio
high,317,354583.0,0.4758
low,1380,0.0,0.23
medium,303,132638.8,0.3603


In [0]:
(
    df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("churn_lakehouse.gold.gold_customer_360")
)

print("✓ gold_customer_360 gravada com sucesso")
spark.sql("SELECT COUNT(*) as total FROM churn_lakehouse.gold.gold_customer_360").display()

✓ gold_customer_360 gravada com sucesso


total
2000


In [0]:
# A feature store contém apenas variáveis numéricas normalizadas
# prontas para o modelo de ML consumir diretamente
# Sem strings, sem datas, sem colunas de negócio — só features

from pyspark.sql.functions import col, when, coalesce, lit

df_features = spark.table("churn_lakehouse.gold.gold_customer_360").select(

    # Identificador
    "customer_id",

    # Target — o que o modelo vai prever
    col("churned").cast("integer").alias("target_churned"),

    # Features de ciclo de vida
    "tenure_days",
    "tenure_months",

    # Features financeiras
    "current_mrr_brl",
    "avg_mrr_brl",
    "total_revenue_brl",
    "total_expansion_brl",
    "total_contraction_brl",
    coalesce("mrr_drop_rate", lit(0.0)).alias("mrr_drop_rate"),

    # Features de pagamento
    "total_invoices",
    "total_billed_brl",
    "total_unpaid_brl",
    coalesce("late_payment_rate", lit(0.0)).alias("late_payment_rate"),
    coalesce("avg_days_late", lit(0.0)).alias("avg_days_late"),
    "max_days_late",
    coalesce("months_with_discount", lit(0.0)).alias("months_with_discount"),

    # Features de suporte
    "nps_score",
    "total_tickets",
    "open_tickets",
    coalesce("open_ticket_rate", lit(0.0)).alias("open_ticket_rate"),
    coalesce("avg_resolution_hours", lit(0.0)).alias("avg_resolution_hours"),

    # Features de engajamento
    "total_logins_90d",
    "avg_logins_per_week",
    "active_features",
    "last_login_days_ago",
    "api_calls_30d",
    "users_active_30d",

    # Features categóricas encodadas
    when(col("plan") == "starter",    1).otherwise(0).alias("plan_starter"),
    when(col("plan") == "growth",     1).otherwise(0).alias("plan_growth"),
    when(col("plan") == "pro",        1).otherwise(0).alias("plan_pro"),
    when(col("plan") == "enterprise", 1).otherwise(0).alias("plan_enterprise"),

    when(col("nps_category") == "detractor", 1).otherwise(0).alias("is_detractor"),
    when(col("nps_category") == "promoter",  1).otherwise(0).alias("is_promoter"),

    when(col("engagement_level") == "low",    1).otherwise(0).alias("engagement_low"),
    when(col("engagement_level") == "medium", 1).otherwise(0).alias("engagement_medium"),
    when(col("engagement_level") == "high",   1).otherwise(0).alias("engagement_high"),

    when(col("last_ticket_type") == "cancelamento", 1).otherwise(0).alias("has_cancellation_ticket"),
    when(col("last_ticket_type") == "bug",          1).otherwise(0).alias("has_bug_ticket"),

    # Score de risco já calculado
    "churn_risk_score",
    "risk_segment",
)

# Verifica o resultado
print(f"Feature Store — {df_features.count():,} clientes, {len(df_features.columns)} features")

# Distribuição do target
df_features.groupBy("target_churned").count().display()

Feature Store — 2,000 clientes, 41 features


target_churned,count
0,1455
1,545


In [0]:
(
    df_features
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("churn_lakehouse.gold.gold_feature_store")
)

print("✓ gold_feature_store gravada com sucesso")
spark.sql("""
    SELECT 
        COUNT(*)                                    AS total_clientes,
        SUM(target_churned)                         AS total_churned,
        ROUND(AVG(target_churned) * 100, 1)         AS churn_rate_pct,
        COUNT(DISTINCT risk_segment)                AS segmentos,
        COUNT(*)                                    AS total_features
    FROM churn_lakehouse.gold.gold_feature_store
""").display()

✓ gold_feature_store gravada com sucesso


total_clientes,total_churned,churn_rate_pct,segmentos,total_features
2000,545,27.3,3,2000
